In [1]:
## Libraries
import numpy as np
import pandas as pd
import librosa
from pathlib import Path


In [2]:
## Parameters
SAMPLING_RATE=16000

## No of frequency bins
## groups the similiar freq sampling rate/n_fft no. of frequencies in each bin 31.25 here
N_FFT=512

## shifts the frame by hop_length data in the audio array
HOP_LENGTH=256
WIN_LEN=512

## Window of different frames from stft,
WINDOW=20
STRIDE=10

In [3]:
## Dataset loading
clean_folder='16k-LP7'
noisy_folder='noisy_dataset'

## Listing all the files in the dataset folder
clean_files=list(Path(clean_folder).rglob('*.wav'))
noisy_files=list(Path(noisy_folder).rglob('*.wav'))


In [4]:
from sklearn.model_selection import train_test_split
train_noisy,test_noisy=train_test_split(noisy_files,test_size=0.3, random_state=110)

In [5]:
def data_generator(train_noisy, batch_size=16, N_FFT=N_FFT,HOP_LENGTH=HOP_LENGTH,WIN_LEN=WIN_LEN, WINDOW=WINDOW):
    while True:
        ## Dictionary for fast cleanup
        clean_dict={}

        ## .stem removes the extension (.wav)
        for file in clean_files:
            clean_dict[file.stem]=file

        X=[]
        Y=[]

        for noisy_path in train_noisy:

            ## base speech name
            base_name=noisy_path.stem.split('_snr')[0]

            if base_name not in clean_dict:
                continue

            clean_path=clean_dict[base_name]

            ## loading audio
            clean,_=librosa.load(clean_path,sr=SAMPLING_RATE)
            noisy,_=librosa.load(noisy_path,sr=SAMPLING_RATE)

            ##STFT
            ## Gives the excel type matrix with different time frames with energy of each bin
            ##  F1|F2|F3
            ##B1 3|2|4
            ##B2 1|1|7
            ##B3 9|6|3

            ## window='hann' fades the amplitude at the starting and the end of our frame
            clean_stft=librosa.stft(clean, n_fft=N_FFT,hop_length=HOP_LENGTH,win_length=WIN_LEN,window='hann')
            noisy_stft=librosa.stft(noisy, n_fft=N_FFT,hop_length=HOP_LENGTH,win_length=WIN_LEN,window='hann')

            ## Magnitude
            clean_mag=np.abs(clean_stft)
            noisy_mag=np.abs(noisy_stft)

            ## Phase
            clean_phase=np.angle(clean_stft)
            noisy_phase=np.angle(noisy_stft)

            ## Phase difference
            phase_diff=clean_phase-noisy_phase

            ## Phase sensitive mask
            ## the noise bins in the clean mag will be probably empty
            ## but the noise bins will be there 
            ## thus the magnitude ratio will be extremely low in the noisy freq bands
            ## we multiply it with the phase diff the phase diff tells time it is seen first in the frame
            psm=(clean_mag/(noisy_mag+1e-8))*np.cos(phase_diff)

            ## Model input
            ## LOG1P adds 1 with all the values of magnitude since the near to zero mag give infinity on log
            ## this scales magnitudes properly considering low mag and high mags
            log_noisy=np.log1p(noisy_mag)

            ## Frame segmentation
            freq_bins, time_frames=log_noisy.shape


            for start in range(0, time_frames-WINDOW +1,STRIDE):
                end=start+WINDOW
                x_seg=log_noisy[:,start:end]
                y_seg=psm[:,start:end]

                

                X.append(x_seg)
                Y.append(y_seg)
                if len(X)==batch_size:
                    ## Transpose for RNN
                    ## this reorganizes the 3d matrix of our features
                    X=np.transpose(X,(0,2,1))
                    Y=np.transpose(Y,(0,2,1))
                    yield np.array(X), np.array(Y)
                    X,Y=[],[]
        print('Total training samples:',len(X))
        print('PSM dataset prepared sucessfully')

In [6]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU,Dense, TimeDistributed,Dropout
import tensorflow as tf

In [7]:
## Input
inp=Input(shape=(20,257))

## GRU
x=GRU(128, return_sequences=True)(inp)
x=GRU(128, return_sequences=True)(x)


## Dense layers (per frame processing)
x=TimeDistributed(Dense(64,activation='relu'))(x)


out=TimeDistributed(Dense(257, activation='sigmoid'))(x)

model=Model(inputs=inp, outputs=out)
optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0)
model.compile(optimizer=optimizer, loss='mse')

In [8]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20, 257)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 20, 128)        │       148,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 20, 128)        │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 20, 64)         │         8,256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 20, 257)        │        16,705 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 272,641 (1.04 MB)

 Trainable params: 272,641 (1.04 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [11]:

early_stop=EarlyStopping(monitor='val_loss',patience=5, restore_best_weights=True)
checkpoint=ModelCheckpoint('best_mosel.h5', monitor='val_loss',save_best_only=True)
lr_reduce=ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=3, min_lr=1e-3)

In [12]:
history=model.fit(data_generator(train_noisy),epochs=10,batch_size=32,callbacks=[early_stop,checkpoint,lr_reduce],steps_per_epoch=45)

Epoch 1/10
44/45 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - loss: 2.0516

c:\Users\hp\anaconda3\Lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss
  current = self.get_monitor_value(logs)
c:\Users\hp\anaconda3\Lib\site-packages\keras\src\callbacks\model_checkpoint.py:276: UserWarning: Can save best model only with val_loss available.
  if self._should_save_model(epoch, batch, logs, filepath):


45/45 ━━━━━━━━━━━━━━━━━━━━ 12s 91ms/step - loss: 2.2322 - learning_rate: 1.0000e-04
Epoch 2/10
 5/45 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - loss: 1.1174

c:\Users\hp\anaconda3\Lib\site-packages\keras\src\callbacks\callback_list.py:145: UserWarning: Learning rate reduction is conditioned on metric `val_loss` which is not available. Available metrics are: loss,learning_rate.
  callback.on_epoch_end(epoch, logs)


44/45 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 0.5811

...

45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - loss: 0.2619 - learning_rate: 1.0000e-04
Epoch 7/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 0.8248

45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 113ms/step - loss: 2.0026 - learning_rate: 1.0000e-04
Epoch 8/10
44/45 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - loss: 0.2850

45/45 ━━━━━━━━━━━━━━━━━━━━ 5s 101ms/step - loss: 0.4659 - learning_rate: 1.0000e-04
Epoch 9/10
44/45 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 0.4121

45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 80ms/step - loss: 0.5263 - learning_rate: 1.0000e-04
Epoch 10/10
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 1.1692

45/45 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - loss: 0.5454 - learning_rate: 1.0000e-04
